# nb_qualidade
Aplica as regras de `config/qualidade.yml` nos dados reais (lakehouse Gold → Files/config/qualidade.yml)
e grava o resultado de cada regra em `ctl_qualidade` (lakehouse Bronze). Não falha por regra quebrada:
o resultado vai para a tabela e o nb_monitoramento transforma as falhas em alerta.

Tipos de regra: chave_unica, nao_nulo, sem_linhas, valores_permitidos, intervalo, minimo_linhas,
sem_timestamp_sem_fuso. Ver o cabeçalho do qualidade.yml.

In [ ]:
grupo = ""              # vazio = todas as regras; ex.: "AUSENCIAS" roda só as do grupo
lote = "manual"

In [ ]:
import os, json, datetime as dt
import pyarrow as pa
import duckdb
from deltalake import DeltaTable, write_deltalake

# Não renomear os lakehouses: o banco do SQL endpoint guarda o nome da criação e a sincronização falha
LH_BRONZE, LH_SILVER, LH_GOLD = "Bronze", "Silver", "Gold"
FUSO_BRT = dt.timezone(dt.timedelta(hours=-3))

_LOCAL = os.environ.get("DW_LOCAL_ROOT")  # usado só em testes fora do Fabric
if _LOCAL:
    def _opts():
        return None
    def caminho(lakehouse, tabela):
        return f"{_LOCAL}/{lakehouse}/{tabela}"
    def sair(valor):
        globals()["_EXIT_VALUE"] = valor
else:
    import notebookutils
    _WS = notebookutils.runtime.context["currentWorkspaceName"]
    def _opts():
        return {"bearer_token": notebookutils.credentials.getToken("storage"),
                "use_fabric_endpoint": "true"}
    def caminho(lakehouse, tabela):
        return f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{lakehouse}.Lakehouse/Tables/{tabela}"
    def sair(valor):
        notebookutils.notebook.exit(valor)

def existe(lakehouse, tabela):
    try:
        DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
        return True
    except Exception:
        return False

def ler(lakehouse, tabela, filtros=None, colunas=None):
    dt_ = DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
    kw = {k: v for k, v in (("filters", filtros), ("columns", colunas)) if v}
    return dt_.to_pyarrow_table(**kw)

def gravar(lakehouse, tabela, dados, modo="append"):
    if modo == "append":
        acrescentar_colunas(lakehouse, tabela, dados.schema)
    kw = {"schema_mode": "overwrite"} if modo == "overwrite" else {}
    write_deltalake(caminho(lakehouse, tabela), dados, mode=modo,
                    storage_options=_opts(), **kw)

def acrescentar_colunas(lakehouse, tabela, schema):
    """Coluna nova na origem (ex.: campo novo da API): a tabela é regravada com a coluna, vazia nas linhas
    antigas, antes do append ou do MERGE. Não depende da versão do deltalake suportar evolução de schema."""
    if not existe(lakehouse, tabela):
        return
    atual = tabela_delta(lakehouse, tabela)
    nomes = {f.name for f in atual.schema().fields}
    novas = [c for c in schema if c.name not in nomes]
    if not novas:
        return
    dados = atual.to_pyarrow_table()
    for c in novas:
        dados = dados.append_column(c, pa.nulls(dados.num_rows, type=c.type))
    write_deltalake(caminho(lakehouse, tabela), dados, mode="overwrite", schema_mode="overwrite",
                    storage_options=_opts())
    print(f"{tabela}: colunas novas {[c.name for c in novas]}")

def tabela_delta(lakehouse, tabela):
    return DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())

def para_arrow(rel):
    """Converte resultado DuckDB em pyarrow.Table (compatível com várias versões)."""
    r = rel.fetch_arrow_table() if hasattr(rel, "fetch_arrow_table") else rel.arrow()
    return r.read_all() if hasattr(r, "read_all") else r

def nome_bronze(tabela_destino):
    """ctl_tabelas guarda o nome base (fato_contas_receber_gs); cada camada recebe o seu prefixo."""
    return f"bronze_{tabela_destino}"

def nome_silver(tabela_destino):
    return f"silver_{tabela_destino}"

def agora_utc():
    return dt.datetime.now(dt.timezone.utc)

def watermark_gold(gold, fonte):
    """Último _dt_carga da Silver já levado para esta Gold (tabela ctl_gold na Bronze). As Golds são
    incrementais por esta marca, e não pelo lote: se uma Gold falhar, a próxima execução retoma dali."""
    if not existe(LH_BRONZE, "ctl_gold"):
        return None
    for r in ler(LH_BRONZE, "ctl_gold").to_pylist():
        if r["gold"] == gold and r["fonte"] == fonte:
            return r["ultimo_dt_carga"]
    return None

def gravar_watermark_gold(gold, fonte, valor):
    linha = pa.Table.from_pylist([{"gold": gold, "fonte": fonte, "ultimo_dt_carga": valor,
                                   "dt_atualizacao": agora_utc()}],
                                 schema=pa.schema([("gold", pa.string()), ("fonte", pa.string()),
                                                   ("ultimo_dt_carga", pa.timestamp("us", tz="UTC")),
                                                   ("dt_atualizacao", pa.timestamp("us", tz="UTC"))]))
    if not existe(LH_BRONZE, "ctl_gold"):
        gravar(LH_BRONZE, "ctl_gold", linha, modo="overwrite")
        return
    (tabela_delta(LH_BRONZE, "ctl_gold")
        .merge(linha, predicate="t.gold = s.gold AND t.fonte = s.fonte", source_alias="s", target_alias="t")
        .when_matched_update_all().when_not_matched_insert_all().execute())

def consulta(nome):
    """Texto de uma consulta da pasta `consultas` (no Fabric: lakehouse Gold → Files/consultas/<nome>.sql,
    copiada de implantacao/consultas pela implantação). Nos testes locais, lê de DW_CONSULTAS."""
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONSULTAS"], f"{nome}.sql"), encoding="utf-8") as arq:
            return arq.read()
    return notebookutils.fs.head(
        f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{LH_GOLD}.Lakehouse/Files/consultas/{nome}.sql", 10_000_000)

def cabecalho(sql):
    """Configuração no topo do .sql, em linhas `-- @chave valor` (ex.: `-- @destino gold_fato_coberturas`)."""
    cfg = {}
    for linha in sql.splitlines():
        linha = linha.strip()
        if linha.startswith("-- @"):
            chave, _, valor = linha[4:].partition(" ")
            cfg[chave] = valor.strip()
    return cfg

def ler_config(nome):
    """Arquivo YAML da pasta config (no Fabric: lakehouse Gold → Files/config/<nome>, copiado de config/ pela
    implantação). Nos testes locais, lê de DW_CONFIG."""
    import yaml
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONFIG"], nome), encoding="utf-8") as arq:
            return yaml.safe_load(arq)
    return yaml.safe_load(notebookutils.fs.head(
        f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{LH_GOLD}.Lakehouse/Files/config/{nome}", 10_000_000))

In [ ]:
LAKEHOUSES = {"Bronze": LH_BRONZE, "Silver": LH_SILVER, "Gold": LH_GOLD}
hoje = dt.datetime.now(FUSO_BRT).date().isoformat()
regras = [r for r in ler_config("qualidade.yml")["regras"] if not grupo or r["grupo"] in (grupo, "GERAL")]

def q(nome):
    return '"' + nome.replace('"', '""') + '"'

def tabelas_da_camada(camada):
    lh = LAKEHOUSES[camada]
    if _LOCAL:
        raiz = os.path.join(os.environ["DW_LOCAL_ROOT"], lh)
        return sorted(n for n in os.listdir(raiz) if os.path.isdir(os.path.join(raiz, n)))
    return sorted(i.name.rstrip("/") for i in notebookutils.fs.ls(
        f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{lh}.Lakehouse/Tables"))

def avaliar(regra):
    """Devolve (linhas com problema, amostra)."""
    camada, _, tabela = regra["tabela"].partition(".")
    tipo = regra["tipo"]
    if tipo == "sem_timestamp_sem_fuso":
        ruins = []
        for t in tabelas_da_camada(camada):
            for campo in tabela_delta(LAKEHOUSES[camada], t).schema().fields:
                if "timestamp_ntz" in str(campo.type).lower():
                    ruins.append(f"{t}.{campo.name}")
        return len(ruins), ruins[:5]
    colunas = regra.get("colunas") or ([regra["coluna"]] if regra.get("coluna") else None)
    dados = ler(LAKEHOUSES[camada], tabela, colunas=colunas)
    if tipo == "minimo_linhas":
        return (0 if dados.num_rows >= int(regra["minimo"]) else 1), [f"{dados.num_rows} linhas"]
    con = duckdb.connect()
    con.execute("SET TimeZone = 'UTC'").fetchall()
    con.register("t", dados)
    if tipo == "chave_unica":
        cols = ", ".join(q(c) for c in regra["colunas"])
        sql = f"SELECT {cols}, count(*) AS repeticoes FROM t GROUP BY {cols} HAVING count(*) > 1"
    elif tipo == "nao_nulo":
        sql = "SELECT * FROM t WHERE " + " OR ".join(f"{q(c)} IS NULL" for c in regra["colunas"])
    elif tipo == "sem_linhas":
        sql = "SELECT * FROM t WHERE " + regra["condicao"].replace("{hoje}", hoje)
    elif tipo == "valores_permitidos":
        valores = ", ".join(repr(v) for v in regra["valores"])
        sql = f"SELECT {q(regra['coluna'])}, count(*) AS linhas FROM t WHERE {q(regra['coluna'])} NOT IN ({valores}) GROUP BY 1"
    elif tipo == "intervalo":
        c = q(regra["coluna"])
        sql = f"SELECT * FROM t WHERE {c} < '{regra['minimo']}' OR {c} > '{regra['maximo']}'"
    else:
        raise ValueError(f"tipo de regra desconhecido: {tipo}")
    n = con.execute(f"SELECT count(*) FROM ({sql})").fetchall()[0][0]
    amostra = [str(l) for l in con.execute(f"{sql} LIMIT 5").fetchall()] if n else []
    con.close()
    return n, amostra

In [ ]:
inicio = agora_utc()
resultados = []
for regra in regras:
    t0 = agora_utc()
    try:
        n, amostra = avaliar(regra)
        status = "OK" if n == 0 else "FALHA"
    except Exception as e:
        n, amostra, status = 0, [str(e)[:500]], "ERRO_EXECUCAO"
    resultados.append({
        "dt_execucao": inicio, "lote": lote, "grupo": regra["grupo"], "regra": regra["nome"],
        "tabela": regra["tabela"], "tipo": regra["tipo"], "severidade": regra["severidade"], "status": status,
        "linhas_problema": int(n), "amostra": json.dumps(amostra, ensure_ascii=False)[:4000],
        "segundos": round((agora_utc() - t0).total_seconds(), 1)})
    print(f"{status:<13} {regra['severidade']:<7} {regra['nome']}: {n}")

schema = pa.schema([("dt_execucao", pa.timestamp("us", tz="UTC")), ("lote", pa.string()), ("grupo", pa.string()),
                    ("regra", pa.string()), ("tabela", pa.string()), ("tipo", pa.string()),
                    ("severidade", pa.string()), ("status", pa.string()), ("linhas_problema", pa.int64()),
                    ("amostra", pa.string()), ("segundos", pa.float64())])
if resultados:
    gravar(LH_BRONZE, "ctl_qualidade", pa.Table.from_pylist(resultados, schema=schema),
           modo="append" if existe(LH_BRONZE, "ctl_qualidade") else "overwrite")

In [ ]:
falhas_erro = [r["regra"] for r in resultados if r["status"] != "OK" and r["severidade"] == "erro"]
falhas_alerta = [r["regra"] for r in resultados if r["status"] != "OK" and r["severidade"] == "alerta"]
sair(json.dumps({"regras": len(resultados), "falhas_erro": falhas_erro, "falhas_alerta": falhas_alerta},
                ensure_ascii=False))